# Phase 6 — DAEAC + pure MCC (Kaggle)

Notebook này chỉ chạy **Minimum Class Confusion (MCC)** với objective `L_source_CE + mu * L_MCC`. Không chạy MK-MMD, focal loss, minority MCC hoặc bất kỳ hybrid nào. Logic huấn luyện nằm trong `scripts/phase6_daeac_mcc/01_train.py` và `src/training/train_daeac_mcc.py`.

In [ ]:
import glob, os, pathlib, shutil, subprocess, sys, yaml

REPO_URL = 'https://github.com/your-user/Best-thesis-in-council.git'
BRANCH = 'another-branch'
WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
if not REPO.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO}
ECG = REPO / 'ecg_thesis'
os.chdir(ECG)
print('cwd =', pathlib.Path.cwd())

## Attach trên Kaggle

Cần attach (1) dataset chứa thư mục `daeac/` với các file `.npz` đã preprocess và (2) model `daeac_base_best.pt`. Đường dẫn checkpoint đã biết của thesis được ưu tiên trước; nếu không tồn tại notebook sẽ tự tìm trong `/kaggle/input`.

In [ ]:
data_roots = [p for p in pathlib.Path('/kaggle/input').glob('**/daeac') if (p / 'mitdb_ds1_daeac.npz').exists()]
assert data_roots, 'Không tìm thấy thư mục daeac chứa mitdb_ds1_daeac.npz trong /kaggle/input'
DATA_SRC = data_roots[0]
DATA_DST = ECG / 'data/processed/phase6_daeac_paper'
DATA_DST.mkdir(parents=True, exist_ok=True)
for src in DATA_SRC.glob('*.npz'):
    shutil.copy2(src, DATA_DST / src.name)

def find_ckpt(preferred, patterns):
    preferred = pathlib.Path(preferred)
    if preferred.exists():
        return str(preferred)
    for pattern in patterns:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0]
    return None

DS1_CKPT = find_ckpt(
    '/kaggle/input/models/thanhvu2357/daeac-base-best/pytorch/default/1/daeac_base_best.pt',
    ['/kaggle/input/**/daeac_base_best.pt'],
)
MITBIH_CKPT = find_ckpt(
    '/kaggle/input/models/thanhvu2357/daeac-base-mitbih-best/pytorch/default/1/daeac_base_mitbih_best.pt',
    ['/kaggle/input/**/daeac_base_mitbih_best.pt', '/kaggle/input/**/daeac_base_mitbih_all_best.pt'],
)
assert DS1_CKPT, 'Không tìm thấy pretrained checkpoint cho source DS1'

print('DATA_SRC =', DATA_SRC)
print('NPZ files =', sorted(p.name for p in DATA_DST.glob('*.npz')))
print('DS1_CKPT =', DS1_CKPT)
print('MITBIH_CKPT =', MITBIH_CKPT)

## Kiểm tra trước khi chạy

Protocol DS1→DS2 cho phép 5 phút đầu DS2 dùng không nhãn để adaptation và toàn bộ DS2 dùng evaluation. Nhãn target không được dataset adaptation trả về trainer.

In [ ]:
!python -u scripts/check_repo.py
!python -u scripts/phase6_daeac_paper/00_validate_data.py --config configs/phase6_daeac_mcc.yaml
!python -u scripts/phase6_daeac_mcc/00_check_protocol.py --config configs/phase6_daeac_mcc.yaml --strict

## Smoke run tùy chọn

Đặt `RUN_SMOKE=True` nếu muốn kiểm tra pipeline. Smoke dùng prefix riêng nên không ghi đè checkpoint full run.

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    !python -u scripts/phase6_daeac_mcc/01_train.py --config configs/phase6_daeac_mcc.yaml --domain-pair ds1_ds2 --init-checkpoint {DS1_CKPT} --epochs 1 --max-source-samples 512 --max-target-samples 512 --max-val-samples 512 --checkpoint-prefix daeac_mcc_smoke

## Tạo config cho 5 domain pairs

DS1→DS2 dùng 5 phút đầu DS2 để adapt. Bốn cặp cross-dataset dùng toàn bộ target. Mỗi cặp có output và checkpoint prefix riêng.

In [ ]:
PAIR_SPECS = {
    'ds1_ds2': ('mitdb_ds1_daeac.npz', 'mitdb_ds2_first5_unlabeled_daeac.npz', 'mitdb_ds2_daeac.npz', 'first5_adapt_full_test', DS1_CKPT),
    'ds1_incart': ('mitdb_ds1_daeac.npz', 'incart_all_daeac.npz', 'incart_all_daeac.npz', 'full_target_transductive', DS1_CKPT),
    'ds1_svdb': ('mitdb_ds1_daeac.npz', 'svdb_all_daeac.npz', 'svdb_all_daeac.npz', 'full_target_transductive', DS1_CKPT),
    'mitbih_incart': ('mitdb_all_daeac.npz', 'incart_all_daeac.npz', 'incart_all_daeac.npz', 'full_target_transductive', MITBIH_CKPT),
    'mitbih_svdb': ('mitdb_all_daeac.npz', 'svdb_all_daeac.npz', 'svdb_all_daeac.npz', 'full_target_transductive', MITBIH_CKPT),
}
RUN_PAIRS = list(PAIR_SPECS)
if any(pair.startswith('mitbih_') for pair in RUN_PAIRS):
    assert MITBIH_CKPT, 'Hai cặp MITBIH cần attach daeac_base_mitbih_best.pt'

with open('configs/phase6_daeac_mcc.yaml', encoding='utf-8') as f:
    base_cfg = yaml.safe_load(f)
# Runtime configs must be direct children of configs/. load_config()
# uses config_path.parents[1] as the project root. A nested folder
# would incorrectly resolve data/... as configs/data/....
runtime_dir = ECG / 'configs'
runtime_dir.mkdir(parents=True, exist_ok=True)
PAIR_CONFIGS = {}
for pair in RUN_PAIRS:
    source, target_adapt, target_test, protocol, ckpt = PAIR_SPECS[pair]
    cfg = yaml.safe_load(yaml.safe_dump(base_cfg))
    cfg['domain_pair'] = pair
    root = 'data/processed/phase6_daeac_paper'
    cfg['data'].update(source_train=f'{root}/{source}', source_eval=f'{root}/{source}', target_unlabeled=f'{root}/{target_adapt}', target_test=f'{root}/{target_test}', target_protocol=protocol)
    cfg['paths']['output_dir'] = f'outputs/phase6_daeac_mcc_{pair}'
    cfg['adaptation']['checkpoint_prefix'] = f'daeac_mcc_{pair}'
    cfg['adaptation']['init_checkpoint'] = ckpt
    path = runtime_dir / f'_runtime_phase6_daeac_mcc_{pair}.yaml'
    path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    PAIR_CONFIGS[pair] = str(path)
    resolved_source = ECG / cfg['data']['source_train']
    resolved_target = ECG / cfg['data']['target_unlabeled']
    assert resolved_source.exists(), f'Missing source: {resolved_source}'
    assert resolved_target.exists(), f'Missing target: {resolved_target}'
    print(pair, '->', path, '| base =', ckpt)

## Full runs — pure MCC

Chỉ nhãn source được dùng khi train. Target loader không trả nhãn. Checkpoint adaptation là epoch cuối.

In [ ]:
for pair in RUN_PAIRS:
    print(f'\n===== TRAIN MCC: {pair} =====', flush=True)
    subprocess.run([sys.executable, '-u', 'scripts/phase6_daeac_mcc/01_train.py', '--config', PAIR_CONFIGS[pair]], check=True)

## Evaluation và visualization

Đánh giá checkpoint `best_dev`, tức epoch có Deep Embedded Validation risk nhỏ nhất. DEV dùng source-validation có nhãn và target-test không nhãn; không đọc nhãn target.

In [ ]:
for pair in RUN_PAIRS:
    cfg = PAIR_CONFIGS[pair]
    out = f'outputs/phase6_daeac_mcc_{pair}'
    ckpt = f'{out}/checkpoints/daeac_mcc_{pair}_best_dev.pt'
    method = f'daeac_mcc_{pair}_best_dev'
    print(f'\n===== EVAL MCC: {pair} =====', flush=True)
    subprocess.run([sys.executable, '-u', 'scripts/phase6_daeac_paper/03_eval.py', '--config', cfg, '--checkpoint', ckpt, '--method-name', method, '--dataset', 'target'], check=True)
    subprocess.run([sys.executable, '-u', 'scripts/phase6_daeac_mcc/06_visualize.py', '--config', cfg, '--checkpoint', ckpt, '--method-name', method, '--dataset', 'target', '--max-samples', '5000'], check=True)

In [ ]:
MCC_OUTPUTS = ' '.join(f'outputs/phase6_daeac_mcc_{pair}' for pair in RUN_PAIRS)
!zip -r /kaggle/working/phase6_daeac_mcc_only_outputs.zip {MCC_OUTPUTS}